1) Perform filtering and normalisation: Explain the reason for chosen your threshold for filtering with appropriate figures and rational for normalisation?

## Reproducibility note

Raw `.h5ad` input files are excluded from Git because they are large single-cell data files. To run this notebook, place the expected sample folders under `data/raw/cscc/` or set the `CSCC_DATA_ROOT` environment variable to the directory containing the sample folders. Full reproducibility still requires access to the original data source and documented download instructions.


In [ ]:
from pathlib import Path
import os
import importlib.util

import scanpy as sc
import scanpy.external as sce
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("CSCC_DATA_ROOT", PROJECT_ROOT / "data" / "raw" / "cscc")).expanduser()
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAMES = [
    "P2_normal",
    "P2_cSCC",
    "P3_normal",
    "P3_cSCC1",
    "P3_cSCC2",
    "P4_normal",
    "P4_cSCC1",
    "P4_cSCC2",
    "P5_normal",
    "P5_cSCC",
]

sample_paths = {
    sample_name: DATA_ROOT / sample_name / "filtered_feature_bc_matrix.h5ad"
    for sample_name in SAMPLE_NAMES
}

missing_files = [path for path in sample_paths.values() if not path.exists()]
if missing_files:
    missing_text = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Missing cSCC input files. Expected each sample folder to contain "
        "filtered_feature_bc_matrix.h5ad.\n\n"
        f"DATA_ROOT currently resolves to: {DATA_ROOT}\n\n"
        "Set CSCC_DATA_ROOT to the directory containing the sample folders, for example:\n"
        "export CSCC_DATA_ROOT=/path/to/local/cscc_data\n\n"
        f"Missing files:\n{missing_text}"
    )

samples = {
    sample_name: sc.read_h5ad(path)
    for sample_name, path in sample_paths.items()
}


In [ ]:
for sample_name, adata_i in samples.items():
    adata_i.obs_names = [f"{sample_name}_{barcode}" for barcode in adata_i.obs_names]
    adata_i.obs_names_make_unique()
    adata_i.var_names_make_unique()
    adata_i.obs["patient"] = sample_name.split("_")[0]
    adata_i.obs["condition"] = "normal" if "normal" in sample_name else "cSCC"
    adata_i.obs["source_sample"] = sample_name

p3_t = sc.concat(
    [samples["P3_cSCC1"], samples["P3_cSCC2"]],
    join="outer",
    label="source_sample_part",
    keys=["P3_cSCC1", "P3_cSCC2"],
)
p3_t.obs["patient"] = "P3"
p3_t.obs["condition"] = "cSCC"
p3_t.obs["source_sample"] = "P3_cSCC"

p4_t = sc.concat(
    [samples["P4_cSCC1"], samples["P4_cSCC2"]],
    join="outer",
    label="source_sample_part",
    keys=["P4_cSCC1", "P4_cSCC2"],
)
p4_t.obs["patient"] = "P4"
p4_t.obs["condition"] = "cSCC"
p4_t.obs["source_sample"] = "P4_cSCC"

adata = sc.concat(
    [
        samples["P2_normal"],
        samples["P2_cSCC"],
        samples["P3_normal"],
        p3_t,
        samples["P4_normal"],
        p4_t,
        samples["P5_normal"],
        samples["P5_cSCC"],
    ],
    join="outer",
    label="sample",
    keys=["P2_normal", "P2_cSCC", "P3_normal", "P3_cSCC", "P4_normal", "P4_cSCC", "P5_normal", "P5_cSCC"],
)

adata


In [ ]:
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="condition",
    multi_panel=True,
    show=False,
)
plt.gcf().savefig(FIGURE_DIR / "01_qc_metrics_by_condition.png", dpi=300, bbox_inches="tight")
plt.show()

sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")


In [ ]:
adata = adata[
    (adata.obs["n_genes_by_counts"] > 500) &
    (adata.obs["n_genes_by_counts"] < 6000) &
    (adata.obs["pct_counts_mt"] < 10) &
    (adata.obs["total_counts"] > 1000) &
    (adata.obs["total_counts"] < 50000)
].copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()

sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
adata = adata[:, adata.var["highly_variable"]].copy()

adata


Cells were filtered based on the QC metric distributions. Cells with fewer than 500 detected genes were excluded as likely low-quality cells, while those with more than 6,000 genes were removed to reduce potential doublets. 
A mitochondrial content threshold of 10% was applied to exclude stressed or dying cells, as indicated by the high-value tail in the distribution. 
Cells with very low total counts (<1,000) were removed due to insufficient RNA content, and very high counts (>50,000) were excluded as they may represent doublets.
These thresholds were chosen to retain the main population of high-quality cells.

2) If you are using dimensionality reduction and integration methods in your pipeline, explain why and how it will affect your downstream anakysis?

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata)

neighbor_rep = "X_pca"

if importlib.util.find_spec("harmonypy") is not None:
    try:
        sce.pp.harmony_integrate(adata, key="patient")
        has_harmony = "X_pca_harmony" in adata.obsm
        valid_harmony = has_harmony and adata.obsm["X_pca_harmony"].shape[0] == adata.n_obs

        if valid_harmony:
            neighbor_rep = "X_pca_harmony"
        else:
            harmony_shape = adata.obsm["X_pca_harmony"].shape if has_harmony else None
            print(
                "Harmony integration did not produce a valid X_pca_harmony matrix; "
                f"using X_pca instead. X_pca_harmony shape: {harmony_shape}; "
                f"expected first dimension: {adata.n_obs}."
            )
    except Exception as exc:
        print(f"Harmony integration failed; using X_pca instead. Details: {exc}")
else:
    print(
        "harmonypy is not installed in this environment; "
        "using unintegrated PCA coordinates for neighbors and UMAP. "
        "Install harmonypy to regenerate integrated results."
    )

sc.pp.neighbors(adata, use_rep=neighbor_rep)
sc.tl.umap(adata)


PCA was used to reduce the dimensionality of the gene expression data, mainly to capture the major sources of variation while getting rid of some of the noise. Working in this lower-dimensional space also makes later steps like building the neighbourhood graph and clustering more manageable. UMAP was then used to visualise the cells in two dimensions, which makes it easier to see how different cell populations are arranged and how they relate to each other.

Since the data comes from multiple patients, there is a possibility that cells could group by patient instead of their biological identity. To address this, Harmony was applied to correct for batch effects between patients. After integration, cells are more likely to group based on their gene expression profiles instead of where they came from. This is important for downstream analysis, especially clustering and cell type identification, because it reduces the chance of picking up patient-driven patterns rather than real biological differences. Overall, these steps help make the results more consistent and easier to interpret.

3) perform neighbourhood, clustering and UMAP analysis
   a) compare at least three different resolutions and which one you will choose and why?
   b) comment how these steps are affecting your Seurat / Anndata object
   hint : look at athe metadata / explore the Anndata object

In [ ]:
neighbor_rep = "X_pca_harmony" if "X_pca_harmony" in adata.obsm else "X_pca"
sc.pp.neighbors(adata, use_rep=neighbor_rep)
sc.tl.umap(adata)

sc.tl.leiden(adata, resolution=0.2, key_added="leiden_0.2")
sc.tl.leiden(adata, resolution=0.5, key_added="leiden_0.5")
sc.tl.leiden(adata, resolution=0.8, key_added="leiden_0.8")

sc.pl.umap(adata, color=["leiden_0.2", "leiden_0.5", "leiden_0.8"], wspace=0.4)

fig = sc.pl.umap(adata, color=["condition", "patient"], return_fig=True, show=False)
fig.savefig(FIGURE_DIR / "02_umap_condition_patient.png", dpi=300, bbox_inches="tight")
plt.show()

print(adata.obs["leiden_0.2"].value_counts())
print(adata.obs["leiden_0.5"].value_counts())
print(adata.obs["leiden_0.8"].value_counts())


a) Leiden clustering was performed at resolutions 0.2, 0.5, and 0.8 to assess the effect of clustering granularity. At resolution 0.2, only a small number of broad cell populations were identified, suggesting under-clustering. At resolution 0.8, the data were partitioned into a larger number of clusters, some of which appeared overly fragmented, suggesting possible over-clustering. Resolution 0.5 was selected for downstream analysis because it provided a balance between identifying biologically distinct populations and avoiding excessive fragmentation.

In [ ]:
print(adata.uns.keys())
print(adata.obsp.keys())
print(adata.obsm.keys())
print(adata.obsm["X_umap"][:5])
print(adata.obs.columns)
adata.obs.head()


b) The neighbourhood step computes a graph of transcriptionally similar cells and stores this in the AnnData object as a connectivity matrix and distance matrix. These are used as the basis for both clustering and UMAP.
UMAP adds a low-dimensional representation of the data to the AnnData object, stored in 'adata.obsm["X_umap"]', which is used for visualisation.
Leiden clustering adds cluster labels to the cell metadata 'adata.obs', allowing each cell to be assigned to a cluster at a given resolution. Different resolutions therefore create different metadata columns corresponding to different clustering granularities.

4) Read the data again and perform all your steps again excluding integration step and compare the UMAP with and without integration. If the UMAP looks different, explain briefly why?

### Deprecated duplicate workflow

This cell previously repeated data loading, preprocessing, integration, and UMAP comparison using hard-coded absolute paths. The reproducible workflow above now performs loading, preprocessing, optional Harmony integration, clustering, UMAP generation, and figure export from the configurable `DATA_ROOT`.


Compared with the integrated analysis, the UMAP without integration is expected to show stronger separation by patient, reflecting batch or sample-specific effects. In contrast, the integrated UMAP should show improved mixing of cells across patients, allowing clustering to reflect biological similarity rather than technical variation. Therefore, integration reduces patient-specific batch effects and changes the structure of the neighbour graph used for UMAP embedding.

5) perform differential expression gene analysis on ontegrated object and annotate as many cluster as you can. confirm your cluster annotation with the markers expression by using plots
   hint :  what normal cell types you can expect in skin? you are free to use manual annotation or any cell anntation tool(for example scType.)

In [ ]:
import scanpy as sc

cluster_key = "leiden_0.5"

if "X_umap" not in adata.obsm:
    neighbor_rep = "X_pca_harmony" if "X_pca_harmony" in adata.obsm else "X_pca"
    sc.pp.neighbors(adata, use_rep=neighbor_rep)
    sc.tl.umap(adata)

sc.pl.umap(adata, color=cluster_key, legend_loc="on data")

sc.tl.rank_genes_groups(
    adata,
    groupby=cluster_key,
    reference="rest",
    method="wilcoxon",
    use_raw=True,
    key_added="rank_genes"
)

cluster_de_genes = {}
for cluster in adata.obs[cluster_key].cat.categories:
    df = sc.get.rank_genes_groups_df(
        adata,
        group=cluster,
        key="rank_genes",
        pval_cutoff=0.05,
        log2fc_min=0.25
    ).sort_values("logfoldchanges", ascending=False)
    
    cluster_de_genes[cluster] = df
    print("
cluster:", cluster)
    display(df.head(10))

marker_dict = {
    "Basal_keratinocyte": ["KRT14", "KRT5", "TP63"],
    "Differentiated_keratinocyte": ["KRT1", "KRT10", "IVL"],
    "Fibroblast": ["COL1A1", "COL1A2", "DCN", "LUM"],
    "Endothelial": ["PECAM1", "VWF", "EMCN"],
    "Pericyte": ["RGS5", "CSPG4", "ACTA2"],
    "T_cell": ["CD3D", "CD3E", "IL7R"],
    "NK_cell": ["NKG7", "GNLY", "KLRD1"],
    "B_cell": ["MS4A1", "CD79A"],
    "Plasma": ["MZB1", "JCHAIN"],
    "Myeloid": ["LYZ", "TYROBP", "FCER1G"],
    "Dendritic": ["FCER1A", "CD1C"],
    "Mast": ["TPSAB1", "KIT", "CPA3"],
    "Melanocyte": ["MLANA", "PMEL", "TYR"]
}

available_markers = {
    cell_type: [gene for gene in genes if gene in adata.raw.var_names]
    for cell_type, genes in marker_dict.items()
}
available_markers = {cell_type: genes for cell_type, genes in available_markers.items() if genes}

sc.pl.dotplot(
    adata,
    var_names=available_markers,
    groupby=cluster_key,
    use_raw=True,
    standard_scale="var"
)

feature_genes = ["KRT14", "KRT1", "COL1A1", "PECAM1", "CD3D", "LYZ", "MLANA", "TPSAB1"]
feature_genes = [gene for gene in feature_genes if gene in adata.raw.var_names]

sc.pl.umap(
    adata,
    color=feature_genes,
    use_raw=True,
    ncols=2
)


In [ ]:
annotation_dict = {
    "0": "Fibroblast",
    "1": "Basal_keratinocyte",
    "2": "Differentiated_keratinocyte",
    "3": "Endothelial",
    "4": "T_cell",
    "5": "Myeloid",
}

adata.obs["cell_type"] = (
    adata.obs[cluster_key]
    .astype(str)
    .map(annotation_dict)
    .fillna("Unannotated")
    .astype("category")
)

sc.pl.umap(adata, color="cell_type", legend_loc="on data")

sc.pl.dotplot(
    adata,
    var_names=available_markers,
    groupby="cell_type",
    use_raw=True,
    standard_scale="var"
)


6) Finally, compare the cell type differences between normal and cSCC and comment on how these changes can lead to cSCC development?
   a) Explore immune and non-immune cells and plot the proportion of cells in normal and cSCC. Are there any significant changes in the composition of cell types between normal and cSCC?
   b) For the immune cell population, describe T cell, B cell and myeloid cell population and their role in cSCC development
   hint : use below mrkers to identify respective cell type

In [ ]:
adata.obs["cell_type"].unique()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cell_type_counts = pd.crosstab(adata.obs["cell_type"], adata.obs["condition"])
cell_type_props = cell_type_counts.div(cell_type_counts.sum(axis=0), axis=1)
print("
Cell type proportions:")
print(cell_type_props)

ax = cell_type_props.T.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5)
)
ax.set_ylabel("Proportion")
ax.set_title("Cell type composition in Normal and cSCC")
ax.legend(title="Cell type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
ax.get_figure().savefig(
    FIGURE_DIR / "03_celltype_composition_normal_vs_cscc.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

immune_types = ["T_cell", "B_cell", "Plasma", "Myeloid", "Dendritic", "NK_cell"]
immune_data = adata[adata.obs["cell_type"].isin(immune_types), :].copy()
immune_counts = pd.crosstab(immune_data.obs["cell_type"], immune_data.obs["condition"])
immune_props = immune_counts.div(immune_counts.sum(axis=0), axis=1)
print("
Immune cell proportions:")
print(immune_props)

immune_ax = immune_props.T.plot(
    kind="bar",
    stacked=True,
    figsize=(7, 4)
)
immune_ax.set_ylabel("Proportion")
immune_ax.set_title("Immune cell composition in Normal and cSCC")
immune_ax.legend(title="Cell type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


(a)
The proportion of cell types differs between normal and cSCC samples. Based on the analysis, epithelial cells are relatively enriched in cSCC compared to normal samples, which is consistent with the expansion of malignant keratinocytes. In addition, fibroblasts and myeloid cells appear to be more abundant in tumor samples, suggesting remodeling of the tumor microenvironment.
In contrast, some normal cell populations are reduced or less prominent in cSCC. These changes indicate that cSCC development is not only driven by epithelial cell proliferation but also involves significant alterations in both stromal and immune cell composition.

(b)
T cells, identified by CD3E expression, play a central role in anti-tumor immunity by recognizing and eliminating malignant cells. In cSCC, T cells are present but may exhibit reduced functionality due to exhaustion or immunosuppressive signals within the tumor microenvironment.
B cells, marked by CD79A, are involved in antibody production and antigen presentation. In this dataset, B cells are present but do not show a dramatic expansion in tumor samples, suggesting a more limited or context-dependent role in cSCC.
Myeloid cells, identified by LYZ expression, are notably enriched in tumor samples. These cells, including tumor-associated macrophages, contribute to an immunosuppressive environment by inhibiting effective T cell responses and promoting tumor growth.
Overall, these findings suggest that immune dysregulation, particularly involving myeloid-driven immunosuppression and T cell dysfunction, plays an important role in cSCC development.